# BERT
Questo quaderno spiega come è stato implementato l'utilizzo di BERT all'interno del nostro progetto.

In [1]:
!pip install -q transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 47.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 KB 10.8 MB/s eta 0:00:00


In [2]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
from scipy.special import softmax
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
import pandas as pd

Come esempio utilizziamo un dataset contenente i tweets precedentemente estratti attraverso lo scraper. Esso contiene una colonna per la data ed una per il tweet già preprocessato. 

In [3]:
twitter = pd.read_csv('./twitter.csv')

In [4]:
twitter

,CONTENT,DATE
0,Belgian politicians debating whose fault is it...,2023-02-04 09:03:06
1,3 MAFIA TURDS WERE WATCHING ME JUST GASSED M...,2023-02-03 10:18:38
2,DESPITE ME MY WIFE ARE LIVING IN DIFFERENT C...,2023-02-01 10:37:21
3,"Libya Italy Tripoli, Libyas oil company, Natio...",2023-01-28 15:19:24
4,"Are you the next Get in touch, and lets talk a...",2023-01-27 09:23:22
...,...,...
2450,Thats the spirit Energy,2019-11-28 00:00:00
2451,Remote participation in todays event on digita...,2019-11-28 00:00:00
2452,"At utility level, 96% of solar, 88% of onshore...",2019-11-28 00:00:00
2453,Shedding light on the essential role of financ...,2019-11-28 00:00:00


Per evitare che il sistema vada out of memory ingombrando eccessivamente la RAM con gli embeddings, il dataset è stato suddiviso in chunks, ciascuno contenente n (in questo caso 10) records. 

In [5]:
tweets = [x for x in twitter.CONTENT.values]
chunks = [tweets[x:x+10] for x in range(0, len(tweets), 10)]

Ciascun chunk viene passato come batch input ad un tokenizer che trasforma le parole in indici numerici corrispondenti secondo un dizionario interno di BERT.
Per ciascun tweet vengono aggiunti due caratteri speciali < CLS > e < SEP >, rispettivamente all'inizio e alla fine. Questi rappresentano l'inizio del testo e la separazione tra due frasi (nel nostro caso abbiamo considerato tutto il testo come un'unica frase, quindi < SEP > indica il termine del testo).

La dimensione massima di ciascun testo tokenizzato è 512 token. Nel caso in cui il testo selezionato contenga meno di 512 elementi, viene effettuato un padding con 0.

In [6]:
inputs = tokenizer(chunks[0], add_special_tokens = True,    
                                truncation = True, padding = "max_length",
                                # return_attention_mask = True, 
                                return_tensors = "pt")

Per ottenere gli embeddings dei tweets è stato utilizzato un modello già addestrato. 
BERT-Base uncased accetta esclusivamente input in cui tutti i caratteri siano lettere minuscole.

E' stata impiegata una versione del modello senza alcun livello specializzato aggiuntivo, in quanto il nostro task richiede esclusivamente di estrapolare gli embedding dal testo.

Per rappresentare ogni tweet è stato selezionato l'embedding del token < CLS >.

In [7]:
model = BertModel.from_pretrained("bert-base-uncased")
outputs = model(**inputs).pooler_output

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In output riceviamo un Tensor di dimensione (n,768), dove n è la dimensione del batch input (e quindi del chunk), mentre 768 corrisponde agli embeddings del testo e rappresenta il numero di neuroni interni presenti nel modello.

In [8]:
outputs.shape

torch.Size([10, 768])

In [9]:
df = twitter.copy().head(10)
df['Tensor'] = [outputs.detach()[i, :] for i in range(outputs.shape[0])]
df['DATE'] = df.DATE.apply(lambda x: pd.to_datetime(x).date())

E' stata inoltre definita una funzione che implementa un meccanismo di attenzione aggiuntivo, da applicare agli embedding dei tweets. 
In particolare, il meccanismo consente di unire l'informazione proveniente da più tweet pubblicati lo stesso giorno.

Nei casi in cui è necessario dare una maggiore importanza al contesto dell'input testuale, la funzione similarità coseno è da preferire al prodotto scalare fra le matrici query e key.

In [10]:
def attention(x, method='cos', seed=None):
  if seed:
    np.random.seed(seed)

  if torch.is_tensor(x):
    x = x.detach()

  W_Q = np.random.random(size=(x.shape[1], x.shape[0]))
  W_K = np.random.random(size=(x.shape[1], x.shape[0]))
  W_V = np.random.random(size=(x.shape[1], x.shape[0]))

  querys = np.dot(x, W_Q)
  keys = np.dot(x, W_K)
  values = np.dot(x, W_V)

  if method == 'dot':
    scores = querys @ keys
  elif method == 'cos':
    scores = cosine_similarity(querys, keys)

  weights = softmax(scores / np.sqrt(x.shape[1]), axis=1)

  return np.sum(weights @ values)


Gli embeddings vengono raggruppati per la data dei tweets a cui fanno riferimento e successivamente il meccanismo di attenzione è applicato ai gruppi.

Il valore di attenzione risultante è inserito in un nuovo dataframe.

In [11]:
new_df = pd.DataFrame(columns=['Date', 'Attention'])
new_df['Date'] = df.DATE.unique()
new_df['Attention'] = new_df.Date.apply(lambda x: attention(torch.stack(tuple(df.loc[df.DATE == x].Tensor.values)), seed=42))

In [12]:
new_df

,Date,Attention
0,2023-02-04,-18.712347
1,2023-02-03,-4.129299
2,2023-02-01,-11.863274
3,2023-01-28,-18.852431
4,2023-01-27,-17.453800
5,2023-01-25,-13.931094
6,2023-01-24,-24.890109
7,2023-01-23,-6.957046
8,2023-01-19,-13.667342
